# Phase 2c: Accuracy reproduction on Llama 3.1 8B

**Goal.** Reproduce KT 2025's reported 98% accuracy on two-digit addition with Llama 3.1 8B (the cleanest model in KT's set).

**Runtime.** A100 40GB **required** (16 GB bf16 weights + KV cache > T4 16GB capacity). Wall time on A100: 5-15 minutes (model download dominates).

**Prompt.** `"The following is a correct addition problem.\n{a}+{b}="` (KT 2025 Table 2; deliberately different from GPT-J/Pythia).

**Outputs to Drive.** `/MyDrive/blackbox_nlp_2026/correctness/llama-3.1-8b.parquet`.

**Pre-registered sanity gate.** `|empirical accuracy - 98%| <= 5pp`. If Llama is the inconvenient case (KT Figure 23), this gate is the cleanest check that decoding works before we look at the harder representation question.

## 1. Standard prelude

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys
PROJECT = '/content/drive/MyDrive/blackbox_nlp_2026'
CODE_DIR = f'{PROJECT}/code_repo'
REPO_URL = 'https://github.com/anshulk-cmu/blackbox-nlp-2026.git'

os.makedirs(PROJECT, exist_ok=True)
if not os.path.exists(CODE_DIR):
    !git clone {REPO_URL} {CODE_DIR}
%cd {CODE_DIR}
!git pull --ff-only

In [ ]:
!pip install -q \
    transformers==4.45.0 \
    huggingface_hub==0.25.1 \
    pandas==2.2.2 \
    pyarrow==17.0.0 \
    accelerate==1.0.1

In [ ]:
from google.colab import userdata
import huggingface_hub
hf_token = userdata.get('HF_TOKEN')
assert hf_token is not None and hf_token.startswith('hf_'), 'HF_TOKEN not set in Colab Secrets.'
huggingface_hub.login(hf_token)
print('HF login OK. Make sure the Llama 3.1 license has been accepted on this HF account.')

In [ ]:
import torch
assert torch.cuda.is_available(), 'GPU required for Phase 2.'
device_name = torch.cuda.get_device_name(0)
memory_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'CUDA: {device_name}')
print(f'Memory: {memory_gb:.1f} GB')
if memory_gb < 30:
    print(f'WARNING: {memory_gb:.1f} GB may be tight for Llama 3.1 8B in bf16. A100 40GB recommended.')

## 2. Load model + tokenizer

In [ ]:
sys.path.insert(0, f'{CODE_DIR}/code')
from accuracy_check import (
    MODEL_CONFIG, load_model_and_tokenizer, run_accuracy_check,
    save_correctness, summary_report, passes_sanity_gate,
    load_intersection_pairs,
)

MODEL_KEY = 'llama-3.1-8b'
print(f'Loading {MODEL_KEY} ({MODEL_CONFIG[MODEL_KEY]["hf_name"]})...')
model, tokenizer = load_model_and_tokenizer(MODEL_KEY, device='cuda')
print(f'  dtype: {next(model.parameters()).dtype}')
print(f'  device: {next(model.parameters()).device}')
print(f'  param count: {sum(p.numel() for p in model.parameters()) / 1e9:.2f} B')
print(f'  CUDA memory after load: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

## 3. Load Phase-1 intersection

In [ ]:
intersection_path = f'{PROJECT}/tokenizer_audit/intersection.json'
assert os.path.exists(intersection_path), \
    f'{intersection_path} missing. Run Phase 1 first.'
pairs = load_intersection_pairs(intersection_path)
print(f'Loaded {len(pairs)} intersection pairs.')

## 4. Run greedy-decode accuracy check

In [ ]:
OUT_DIR = f'{PROJECT}/correctness'
os.makedirs(OUT_DIR, exist_ok=True)
OUT_PATH = f'{OUT_DIR}/{MODEL_KEY}.parquet'

df = run_accuracy_check(
    model, tokenizer, pairs, MODEL_KEY,
    batch_size=16,
    progress_every=10,
    save_every=200,
    partial_path=OUT_PATH,
)
save_correctness(df, OUT_PATH)
print(f'\nSaved {len(df)} rows to {OUT_PATH}')

## 5. Summary report and sanity gate

In [ ]:
print(summary_report(df, MODEL_KEY))
print()
if passes_sanity_gate(df, MODEL_KEY, tolerance_pp=5.0):
    print('PASS: empirical accuracy within 5pp of KT. Proceed to Phase 3.')
else:
    print('FAIL: empirical accuracy more than 5pp from KT. Halt and re-check.')
    print('For Llama specifically, check that the prompt template includes the')
    print('newline (different from GPT-J/Pythia) and that bf16 + left-padding work.')

## 6. Cleanup

In [ ]:
del model
import gc
gc.collect()
torch.cuda.empty_cache()
print(f'Freed. CUDA memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB')